In [ ]:
from mlflow.genai.judges import make_judge
from typing import Literal
import mlflow
from typing import cast
from mlflow.entities import Expectation, Trace, AssessmentSource , AssessmentSourceType, SpanType
MLFLOW_TRACKING_URI = "http://100.113.186.28:5000"
EXPERIMENT_NAME = "vds-agent-validation"
EXPERIMENT_ID = "6"
TRACE_FILTER = 'trace.text LIKE "%I want%"'

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

In [ ]:
traces = cast(
    list[Trace],
    mlflow.search_traces(
        locations=[EXPERIMENT_ID],
        filter_string=TRACE_FILTER,
        return_type='list'
    )
)

In [ ]:
test_trace = next(
    filter(lambda x: x.info.trace_id == 'tr-c073a25e661ca5b9a4be705fb2bd04c9', traces)
)
test_trace

In [ ]:
ignore_tool_list = [
    'get_member_information',
    'delegate_task_to_member',
    'get_available_worker_tools',
    'get_available_models',
    'spawn_and_run_worker'
]
def extract_trace_steps(trace: Trace):
    spans = trace.data.spans
    results = []
    i = 0
    n = len(spans)
    while i < n:
        span = spans[i]
        if span.span_type == SpanType.TOOL and span.name in ignore_tool_list:
            i += 1
            continue

        if span.span_type == SpanType.TOOL and span.name == "spawn_and_run_worker":
            tool_args = span.inputs or {}

            worker_info: dict = {
                "worker_name": tool_args.get("agent_name"),
                "description": tool_args.get("agent_description"),
                "toolset": tool_args.get("tool_names", []),
                "agent_output": str(span.outputs),
                "steps": []
            }

            i += 1
            while i < n:
                next_span = spans[i]

                if (
                    next_span.span_type == SpanType.TOOL
                    and next_span.name == "spawn_and_run_worker"
                ):
                    break

                if next_span.span_type == SpanType.TOOL:
                    worker_info["steps"].append({
                        "tool": next_span.name,
                        "args": str(next_span.inputs)[:300] if next_span.inputs else "",
                        "output": str(next_span.outputs)[:500] if next_span.outputs else "",
                        "step_index": i + 1
                    })

                i += 1

            results.append(worker_info)
            continue
        i += 1
        
    return {
        "trace_id": trace.info.trace_id,
        "workers": results
    }

def format_trace(trace: Trace):
    parsed = extract_trace_steps(trace)

    lines = []
    lines.append(f"TRACE ID: {parsed['trace_id']}")
    lines.append("=" * 60)

    for w_idx, worker in enumerate(parsed["workers"], 1):
        lines.append(f"\n[Worker {w_idx}] {worker['worker_name']}")
        
        for step in worker["steps"]:
            lines.append(f"  [Step {step['step_index']}] Tool: {step['tool']}")
            lines.append(f"     args: {step['args']}")
            lines.append(f"     output: {step['output']}")
        
        lines.append(f"     agent output: {worker['agent_output']}")

    return "\n".join(lines)

def get_record_based_on_trace(
    data_df: list[dict],
    trace: Trace
): 
    """
    Get the corresponding record based on trace's session id
    """
    trace_input_preview = cast(str, trace.info.request_preview).strip('"').strip("'").replace('\\n', '')
    
    filter_data_record = next(
        filter(
            lambda x: x['inputs']['user_demand'].replace('\n', '') == trace_input_preview, data_df
        )
    )
    
    return filter_data_record

def build_prompt(trace: Trace, eval_record: list[dict]):
    trace_text = format_trace(trace)
    expected = get_record_based_on_trace(eval_record, trace)

    prompt = f"""
You are analyzing an agent execution trace to detect when the correct answer was already found.

=== TRACE ===
{trace_text}

=== EXPECTED RESPONSE ===
{expected['expectations']['expected_response']}

=== TASK ===
Your goal is to locate the EARLIEST step or many steps that constitute the result (All video ids and all the points in the expected response) where the system already had enough information to produce the expected answer.

IMPORTANT:
- A "step" corresponds to the numbered step in the trace
- Focus on TOOL args/outputs (not thoughts)
- Match based on semantic correctness (video_id, segment, or equivalent info)

=== OUTPUT FORMAT (STRICT) ===
Return a JSON object with:
```json
{{
  "trace_id": "{trace.info.trace_id}",
  "answer_found": {{
    "step_index": int,
    "worker": "agent_name",
    "tools_involved": [
      {{
        "tool": "tool_name",
        "step": int,
        "reason": "why this tool contributes to the answer"
      }}
    ],
    "evidence": "does the agent output contain the expected response?, what tool/set of tools led to the discovery."
  }},
  "final_judgement": "explanation"
}}
```json
=== RULES ===
- ALWAYS pick the earliest valid step
- If multiple tools are needed, include all contributing steps
- If answer is never found, return step_index = -1
- Be precise: do not guess beyond tool outputs
"""

    return prompt

In [ ]:
from pathlib import Path
import json
BASE_DIR = Path("/home/tinhanhnguyen/Desktop/HK8/Capstone/CAPSTONE_PROJECT/videodeepsearch")
EVAL_RECORDS_PATH = BASE_DIR / "local/mlflow_eval_records.json"
OUTPUT_DIR = BASE_DIR / "test/notebooks/analysis_results"


with open(EVAL_RECORDS_PATH, "r") as f:
    eval_records = json.load(f)



prompts = []

for trace in traces:  # sample first 5
    prompts.append({
        "trace_id": trace.info.trace_id,
        "prompt": build_prompt(trace, eval_records)
    })

len(prompts)

In [ ]:
for prompt in prompts:
    trace_id = prompt['trace_id']
    prompt_text =  prompt['prompt']
    file_path = Path('/home/tinhanhnguyen/Desktop/HK8/Capstone/CAPSTONE_PROJECT/videodeepsearch/test/notebooks/prompts') / f"{trace_id}.txt"
    with open(file_path, 'w') as f:
        f.write(f"Trace ID: {trace_id}\n")
        f.write(f"Prompt: {prompt_text}\n")

In [ ]:

import requests
import json
import re
from pathlib import Path

def extract_json(text):
    pattern = r"```json\s*(.*?)\s*```"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1)
    return text 


url = "http://localhost:8080/completion"
output_dir = Path("prompts_results")
output_dir.mkdir(exist_ok=True)

responses = []
retry_limit = 2

for prompt in prompts:
    trace_id = prompt['trace_id']
    prompt_text = prompt['prompt']
    
    output = None
    success = False
    
    for attempt in range(retry_limit):
        try:
            response = requests.post(
                url, 
                json={
                    "prompt": prompt_text,
                    "temperature": 0.7
                },
                timeout=750
            )
            
            response.raise_for_status()
            
            raw_content = response.json().get("content", "")
            clean_raw_content = extract_json(raw_content)

            try:
                output_dict = json.loads(clean_raw_content)
                output = clean_raw_content
                success = True
                break 
            except json.JSONDecodeError as e:
                print(f"Attempt {attempt + 1}: Failed to parse LLM output as JSON: {e}")
                print(raw_content)
                output = raw_content # Keep the raw text anyway
                
        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt + 1}: Connection error: {e}")
    
    # Save the result
    file_path = output_dir / f"{trace_id}.txt"
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(f"Response: {output if output else 'FAILED'}\n")

    if not success:
        print(f"Final Failure for Trace ID: {trace_id}")